# Trabalho Final PACD
**Título:** Eficiência de Recursos nas PME Europeias: Fatores Determinantes da Adoção e da sua Intensidade  
**Fonte de dados:** Flash Eurobarometer 549 (Junho 2024) — GESIS ZA8869  
**Amostra:** PME dos 27 países da UE + Reino Unido

---

## Setup

In [1]:
import pyreadstat
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import warnings
warnings.filterwarnings('ignore')
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
%matplotlib inline

# ── Paleta de cores global ────────────────────────────────────────────────────
BLUE       = '#4C6EFF'   # azul principal
LIGHT_BLUE = '#A3C1FF'   # azul claro
ORANGE     = '#FF7043'   # laranja — alertas / destaque
GREEN      = '#2E7D5E'   # verde — Q8 / apoios
BG         = '#F9F9F9'   # fundo dos gráficos

---
# Secção 1 — Extração e Preparação dos Dados

> Esta secção cobre o carregamento do dataset, a recoding dos valores em falta, a aplicação dos filtros de âmbito e a seleção das variáveis relevantes para o estudo.

## 1.1 Carregamento do Dataset

In [2]:
# Carregamento do ficheiro de dados original
# Fonte: Flash Eurobarometer 549 (Junho 2024) — GESIS ZA8869
# Formato: SPSS .sav com códigos numéricos
# apply_value_formats=False → mantém os códigos numéricos originais (mais estável para filtros)
df_raw, meta = pyreadstat.read_sav('../data/raw/initial_data.sav', apply_value_formats=False)
print(f"Dataset carregado: {df_raw.shape[0]:,} linhas × {df_raw.shape[1]} colunas")

# ── Recoding dos valores DK/NA (convenção SPSS do Eurobarómetro) ──────────────
# 997/998/999 = Don't know / No answer → NaN
# 999997/999998/999999 = mesmos códigos para variáveis de valor elevado
DK_NA = {997.0, 998.0, 999.0, 999997.0, 999998.0, 999999.0}
df_raw = df_raw.copy()
for col in df_raw.select_dtypes(include='number').columns:
    df_raw[col] = df_raw[col].replace({k: np.nan for k in DK_NA})

print(f"Recoding DK/NA concluído.")
df_raw.head()

Dataset carregado: 18,159 linhas × 236 colunas
Recoding DK/NA concluído.


,studyno,doi,version,edition,survey,uniqid,serialid,ipscntry,country,isocntry,nace_a,nace_b,d1b,scr10a,scr10b,scr10,scr11a,scr11b,scr12,scr12a,scr12b,scr13a,scr14,scr16,q1.1,q1.2,q1.3,q1.4,q1.5,q1.6,q1.7,q1.8,q1.9,q1.10,q1.11,q1.12,q1t,q2.1,q2.2,q2.3,q2.4,q2.5,q2.6,q2.7,q2.8,q2.9,q2.10,q2.11,q2.12,q2t,q3,q4,n1a,n1b,q5.1,q5.2,q5.3,q5.4,q5.5,q6.1,q6.2,q6.3,q6.4,q6.5,q6.6,q6.7,q6.8,q6.9,q7.1,q7.2,q7.3,q7.4,q7.5,q7.6,q7.7,q7.8,q7.9,q7.10,q7.11,q7.12,q7.13,q8.1,q8.2,q8.3,q8.4,q8.5,q8.6,q8.7,q8.8,q8.9,q8.10,q8.11,q14,q15.1,q15.2,q15.3,q15.4,q15.5,q15.6,q15.7,n2,n3,q9,q10,q13.1,q13.2,q13.3,q13.4,q13.5,dx1.1,dx1.2,dx1.3,dx1.4,dx1.5,dx1.6,dx1.7,dx1.8,dx1.9,dx3.1,dx3.2,dx3.3,dx3.4,dx3.5,dx3.6,dx3.7,dx4.1,dx4.2,dx4.3,dx4.4,dx4.5,dx4.6,dx4.7,dx5,dx5r,d12be,d12be_r,d12at,d12at_r,d12bg,d12cy,d12cz,d12dk,d12ee,d12de,d12gr,d12gr_r,d12es,d12es_r,d12fi,d12fr,d12fr_r,d12hu,d12ie,d12it,d12it_r,d12lt,d12lu,d12lv,d12nl,d12nl_r,d12pl,d12pl_r,d12pt,d12ro,d12se_r,d12si,d12si_r,d12sk,d12mt,d12hr,d12al,d12is,d12me_r,d12mk,d12no,d12rs,d12rs_r,d12tr,d12gb,d12us,d12us_r,d12md,d12ch,eu6,eu9,eu10,eu12_2,eu_nms3,eu15,euroz20,euronz20,eu_nms10,eu_nms12,eu_nms13,eu25,eu27,eu_la6,eucc5,v_noneu_us,v_efta,w1,w5,w6,w7,w9,w10,w11,w13,w14,w24,w87,w94,w102,w103,w100,w91,w109,w113,w1_sme,w5_sme,w6_sme,w7_sme,w9_sme,w10_sme,w11_sme,w13_sme,w14_sme,w24_sme,w87_sme,w94_sme,w102_sme,w103_sme,w100_sme,w91_sme,w109_sme,w113_sme
0,8869.0,doi:10.4232/1.14493,1.0.0 (2025-03-12),1.0,549.0,34000001.0,1.0,34.0,37.0,AL,5.0,2.0,3.0,160.0,NaN,3.0,3.0,3.0,1.0,9998.0,1.0,1.0,8.0,3.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,4.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,3.0,NaN,NaN,3.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,3.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,1.0,3.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,7.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,11.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.065564,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.027545,NaN,0.027793,0.063834,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.025719,NaN,0.025947
1,8869.0,doi:10.4232/1.14493,1.0.0 (2025-03-12),1.0,549.0,34000002.0,2.0,34.0,37.0,AL,12.0,4.0,11.0,75.0,NaN,3.0,1.0,1.0,1.0,2011.0,NaN,2.0,8.0,2.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,3.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,3.0,1.0,3.0,2.0,2.0,1.0,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,3.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,1.0,0.0,0.0,0.0,0.0,NaN,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,11.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.033504,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.014075,NaN,0.014202,0.032620,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.013142,NaN,0.013259
2,8869.0,doi:10.4232/1.14493,1.0.0 (2025-03-12),1.0,549.0,34000003.0,3.0,34.0,37.0,AL,5.0,2.0,3.0,280.0,NaN,4.0,1.0,2.0,1.0,2010.0,NaN,1.0,2.0,3.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,4.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,4.0,NaN,NaN,3.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,3.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,3.0,3.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,N

In [3]:
# Verificar os valores únicos de isocntry — confirmar se UK aparece como 'GB' ou 'UK'
print("Valores únicos de isocntry:")
print(sorted(df_raw['isocntry'].dropna().unique()))
print(f"\nTotal países: {df_raw['isocntry'].nunique()}")

Valores únicos de isocntry:
['AL', 'AT', 'BE', 'BG', 'CH', 'CY', 'CZ', 'DE', 'DK', 'EE', 'ES', 'FI', 'FR', 'GB', 'GR', 'HR', 'HU', 'IE', 'IS', 'IT', 'LT', 'LU', 'LV', 'MD', 'ME', 'MK', 'MT', 'NL', 'NO', 'PL', 'PT', 'RO', 'RS', 'SE', 'SI', 'SK', 'TR', 'US']

Total países: 38


## 1.2 Filtro Geográfico — UE + Reino Unido

> O Reino Unido é mantido porque participou no inquérito apesar do Brexit. Os países candidatos e EFTA são excluídos por não integrarem o mercado único.

In [ ]:
# Filtro geográfico: reter apenas os 27 países da UE + Reino Unido
# Nota: o ficheiro usa ISO-2 standard — 'GR' (não 'EL') para a Grécia, 'GB' (não 'UK') para o RU
eu_uk = [
    'AT','BE','BG','HR','CY','CZ','DK','EE','FI','FR',
    'DE','GR','HU','IE','IT','LV','LT','LU','MT','NL',
    'PL','PT','RO','SK','SI','ES','SE',
    'GB'   # Reino Unido
]

df = df_raw[df_raw['isocntry'].isin(eu_uk)].copy()
print(f"Dataset original:        {df_raw.shape[0]:,} empresas")
print(f"Após filtro EU+UK:        {df.shape[0]:,} empresas")
print(f"Removidas:                {df_raw.shape[0] - df.shape[0]:,} empresas")
print(f"\nPaíses na amostra: {df['isocntry'].nunique()}")
print(df['isocntry'].value_counts())

## 1.3 Filtro PME — Empresas com menos de 250 FTE

> De acordo com a Recomendação Europeia 2003/361/CE, a PME é definida como tendo menos de 250 FTE. As categorias 250–499 e 500+ são excluídas por estarem fora do âmbito do estudo.

In [5]:
# scr10 (dimensão FTE):
#   1 = Micro   (1–9 FTE)
#   2 = Pequena (10–49 FTE)
#   3 = Média   (50–249 FTE)
#   4 = Grande  (250–499 FTE)  → excluída
#   5 = Muito grande (500+)    → excluída

size_map = {
    1.0: 'Micro (1–9)',
    2.0: 'Pequena (10–49)',
    3.0: 'Média (50–249)',
    4.0: 'Grande (250–499)',
    5.0: 'Muito grande (500+)',
}

print("Distribuição por dimensão ANTES do filtro PME:")
print(df['scr10'].map(size_map).value_counts())
print(f"Total: {len(df):,}")

df = df[df['scr10'].isin([1.0, 2.0, 3.0])].copy()

print(f"\nDistribuição por dimensão APÓS filtro PME:")
print(df['scr10'].map(size_map).value_counts())
print(f"\nAmostra final PME: {df.shape[0]:,} empresas")
n_before_pme = len(df_raw[df_raw['isocntry'].isin(eu_uk)])
print(f"Grandes empresas removidas: {n_before_pme - df.shape[0]:,}")

Distribuição por dimensão ANTES do filtro PME:
scr10
Micro (1–9)            5717
Pequena (10–49)        4767
Média (50–249)         2509
Grande (250–499)        636
Muito grande (500+)     312
Name: count, dtype: int64
Total: 13,949



Distribuição por dimensão APÓS filtro PME:
scr10
Micro (1–9)        5717
Pequena (10–49)    4767
Média (50–249)     2509
Name: count, dtype: int64

Amostra final PME: 12,993 empresas
Grandes empresas removidas: 956


## 1.4 Seleção e Documentação das Variáveis Chave

In [6]:
# Seleção das variáveis relevantes para o estudo
#
# ── Variável Dependente ───────────────────────────────────────────────────────
# Q1.1–Q1.9  → 9 práticas de eficiência de recursos (binário)
#               → ÍNDICE DE INTENSIDADE (0–9) construído na Secção 3
#
# ── Variáveis Independentes — Estruturais ─────────────────────────────────────
# isocntry   → País (ISO-2)
# nace_b     → Setor de atividade (4 categorias: 1=Manufatura, 2=Indústria, 3=Retalho, 4=Serviços)
# scr10      → Dimensão FTE (1=Micro, 2=Pequena, 3=Média)
#
# ── Variáveis Independentes — Características da Empresa ──────────────────────
# scr12      → Antiguidade categórica (1=Antes 2016, 2=2016-18, 3=2019-23, 4=Após 2023)
# scr11a     → Evolução do nº de empregados (1=Aum.≥10%, 2=Aum.<10%, 3=Igual, 4=Dim.)
# scr13a     → Evolução do volume de negócios (1=Aum.≥10%, 2=Aum.<10%, 3=Igual, 4=Dim.)
# scr14      → Volume de negócios anual (faixas)
#
# ── Variáveis Independentes — Financeiras / Estratégicas ──────────────────────
# q3         → Impacto nos custos de produção das ações de eficiência
# q4         → % da receita investida em ações ambientais
#
# ── Barreiras à Eficiência (Q7) ───────────────────────────────────────────────
# q7.1–q7.12 → 12 barreiras (binárias: 1=mencionada, NaN=não mencionada)
#
# ── Pesos de inquérito ────────────────────────────────────────────────────────
# w1_sme     → Peso pós-estratificação para PME (descritivas ponderadas)
#

key_vars = [
    # Estruturais
    'isocntry', 'nace_b', 'scr10',
    # Características da empresa
    'scr12', 'scr11a', 'scr13a', 'scr14',
    # Financeiras / estratégicas
    'q3', 'q4',
    # Variável dependente
    'q1.1','q1.2','q1.3','q1.4','q1.5',
    'q1.6','q1.7','q1.8','q1.9',
    # Barreiras
    'q7.1','q7.2','q7.3','q7.4','q7.5',
    'q7.6','q7.7','q7.8','q7.9','q7.10',
    'q7.11','q7.12',
    # Peso
    'w1_sme',
]

# Reter apenas variáveis que existam no dataset (protege contra versões do ficheiro)
key_vars = [v for v in key_vars if v in df.columns]
missing_vars = [v for v in key_vars if v not in df.columns]
if missing_vars:
    print(f"⚠ Variáveis não encontradas: {missing_vars}")

df_key = df[key_vars].copy()
print(f"df_key: {df_key.shape[0]:,} linhas × {df_key.shape[1]} colunas")
print(f"Variáveis seleccionadas: {key_vars}")

df_key: 12,993 linhas × 30 colunas
Variáveis seleccionadas: ['isocntry', 'nace_b', 'scr10', 'scr12', 'scr13a', 'scr14', 'q3', 'q4', 'q1.1', 'q1.2', 'q1.3', 'q1.4', 'q1.5', 'q1.6', 'q1.7', 'q1.8', 'q1.9', 'q7.1', 'q7.2', 'q7.3', 'q7.4', 'q7.5', 'q7.6', 'q7.7', 'q7.8', 'q7.9', 'q7.10', 'q7.11', 'q7.12', 'w1_sme']


In [7]:
# ── Dicionários de labels — reutilizados em toda a análise ────────────────────

q1_map = {
    'q1.1': 'Poupar água',
    'q1.2': 'Poupar energia',
    'q1.3': 'Energia renovável',
    'q1.4': 'Poupar materiais',
    'q1.5': 'Fornecedores mais verdes',
    'q1.6': 'Minimizar resíduos',
    'q1.7': 'Vender resíduos',
    'q1.8': 'Reciclar internamente',
    'q1.9': 'Eco-design',
}

q7_map = {
    'q7.1':  'Complexidade admin./legal',
    'q7.2':  'Dificuldade em adaptar legislação',
    'q7.3':  'Requisitos técnicos desatualizados',
    'q7.4':  'Dificuldade em escolher ações',
    'q7.5':  'Custo das ações ambientais',
    'q7.6':  'Falta de expertise ambiental',
    'q7.7':  'Falta de oferta de materiais/peças',
    'q7.8':  'Falta de procura por prod. eficientes',
    'q7.9':  'Complexidade de rotulagem ambiental',
    'q7.10': 'Requisitos complexos de reporte',
    'q7.11': 'Outra',
    'q7.12': 'Nenhuma',
}
# Filtrar para barreiras que existam no df_key
q7_map = {k: v for k, v in q7_map.items() if k in df_key.columns}

# Mapeamentos para variáveis numéricas
size_map = {
    1.0: 'Micro (1–9)',
    2.0: 'Pequena (10–49)',
    3.0: 'Média (50–249)',
}
size_order = [1.0, 2.0, 3.0]
size_short = ['Micro\n(1–9)', 'Pequena\n(10–49)', 'Média\n(50–249)']

sector_map = {
    1.0: 'Manufatura (C)',
    2.0: 'Indústria (B/D/E/F)',
    3.0: 'Retalho (G)',
    4.0: 'Serviços (H–M)',
}
sector_order = [1.0, 2.0, 3.0, 4.0]
sector_short  = ['Manufatura', 'Indústria', 'Retalho', 'Serviços']

print("Labels definidos com sucesso.")
print(f"  q1_map:     {len(q1_map)} práticas")
print(f"  q7_map:     {len(q7_map)} barreiras")

Labels definidos com sucesso.
  q1_map:     9 práticas
  q7_map:     12 barreiras
